## correct 데이터에서 unfortunately... 등 제거

In [ ]:
import random
import re
import pandas as pd

def delete_comma_at_the_end(df):
    df['previous_steps'] = df['previous_steps'].apply(lambda l: [x.strip(",") for x in l])
    df['current_step'] = df['current_step'].apply(lambda x: x.strip(","))
    return df

def get_paraphrased_text(text):
	"""
	입력된 텍스트와 인덱스(0~18)를 받아, 
	해당하는 단 하나의 Paraphrasing된 문자열을 반환하는 함수
	"""
	# 1. 인덱스 유효성 검사
	index = random.randint(0, 18)

	# 2. 정규표현식 파싱
	pattern = r"According to Passage (\d+), (.+?)\. \(Attribution\)"
	match = re.search(pattern, text)
	
	if not match:
		print(f"Error: 입력 텍스트 형식이 올바르지 않습니다. -> {text}")
		return None

	p_id = match.group(1)
	content = match.group(2)

	# 3. 사용할 Content 결정 (Group C인 13번부터는 대문자 변환 필요)
	if index >= 13:
		# 문장 앞으로 오므로 첫 글자 대문자 처리
		final_content = content[0].upper() + content[1:]
	else:
		# 문장 뒤에 오므로 원본 유지
		final_content = content

	# 4. 인덱스별 템플릿 정의 (Dictionary)
	templates = {
		# --- Group A: Preposition (전치사구) ---
		0: f"According to Passage {p_id}, {final_content}. (Attribution)",
		1: f"Based on Passage {p_id}, {final_content}. (Attribution)",
		2: f"Refer to Passage {p_id}, {final_content}. (Attribution)",
		3: f"As stated in Passage {p_id}, {final_content}. (Attribution)",
		4: f"From the information in Passage {p_id}, {final_content}. (Attribution)",
		5: f"As you can see in Passage {p_id}, {final_content}. (Attribution)",
		
		# --- Group B: Verb Phrases (주어+동사) ---
		6: f"Passage {p_id} states that {final_content}. (Attribution)",
		7: f"Passage {p_id} confirms that {final_content}. (Attribution)",
		8: f"Passage {p_id} reveals that {final_content}. (Attribution)",
		9: f"Passage {p_id} indicates that {final_content}. (Attribution)",
		10: f"It is mentioned in Passage {p_id} that {final_content}. (Attribution)",
		11: f"We can find in Passage {p_id} that {final_content}. (Attribution)",
		12: f"Looking at Passage {p_id}, we see that {final_content}. (Attribution)",

		# --- Group C: Inverted (도치/후수식) ---
		13: f"{final_content}, according to Passage {p_id}. (Attribution)",
		14: f"{final_content}, based on Passage {p_id}. (Attribution)",
		15: f"{final_content}, as shown in Passage {p_id}. (Attribution)",
		16: f"{final_content}, as stated in Passage {p_id}. (Attribution)",
		17: f"{final_content}, which is supported by Passage {p_id}. (Attribution)",
		18: f"{final_content}, as referencing Passage {p_id}. (Attribution)"
	}

	return templates[index]

def more_than_1_passage_filter(df):
	print("1. More than 1 passage filter")
	filter1 = []
	for i, row in df.iterrows():
		is_wrong=False
		for step in row["ideal_steps"]:
			passage_refs = []
			matches = re.findall(r"Passage\s+\d+", step)
			passage_refs.extend(matches)

			# Passage 언급이 2개 이상이면 제외
			if len(passage_refs) >= 2:
				is_wrong = True
				# print(f"prev Row {i}: {step}.")
				break

		if is_wrong:
			continue
		filter1.append(row)

	filter1_df = pd.DataFrame(filter1)
	print(f"{len(filter1_df)} after passage count filtering")
	return filter1_df

def says_no_information_filter(df):
	filter_list = [
		"do not mention", "does not mention", "do not provide", "does not provide",
		"do not state", "does not state", "not provided", "not mentioned", "not stated", 
		"not explicitly mentioned", "not explicitly provided", "not explicitly stated", 
		"not present in", "no information in the provided", "not explicitly state", 
		"not explicitly mention", "does not specify", "do not specify", "is not specified", 
		"there is no explicit information", "there is no information",
		"is not directly mentioned", "is not directly stated", "is not directly provided", 
		"there's no direct information", "is not found in", "do not directly mention", "does not directly mention",
	]
	print("2. Says no information filtering")
	filter2 = []
	for i, row in df.iterrows():
		flag = False
		for step in row['ideal_steps']:
			step_lower = step.lower()
			# 하나라도 금지 문구가 포함되어 있으면 플래그 설정
			if any(phrase in step_lower for phrase in filter_list):
				flag = True
				break

		if not flag:
			filter2.append(row)

	filter2_df = pd.DataFrame(filter2)
	print(len(filter2_df))
	return filter2_df

def previous_steps_paraphrase_filter(df):
	rows = []
	for i, row in df.iterrows():
		new_previous_steps = []
		is_wrong = False
		for j, step in enumerate(row['previous_steps']):
			if step.endswith("(Logical)"):
				new_previous_steps.append(step)
				continue
			if get_paraphrased_text(step):
				new_previous_steps.append(f"Step {j+1}: {get_paraphrased_text(step)}")
			else:
				is_wrong = True
				break
		if is_wrong:
			# print(f"Warning: Some steps in row {i} could not be paraphrased correctly.")
			continue
		row['previous_steps'] = new_previous_steps
		rows.append(row)
	print(f"{len(rows)} after previous steps paraphrase filtering")
	return pd.DataFrame(rows)

def current_step_paraphrase_filter(df):
	for i, row in df.iterrows():
		if row['current_step'].endswith("(Logical)"):
			continue
		
		if get_paraphrased_text(row['current_step']):
			df.at[i, 'current_step'] = f"Step {j+1}: {get_paraphrased_text(row['current_step'])}"
			
	return df


# contradictory, unsupported: 통과
# redundancy, logical_fallacy, information_miss, inefficiency: previous steps만 해야함
# off_topic, overthinking, correct_improved: 둘 다 해야함

for name in ["correct_improved"]:
	df = pd.read_json(f"/workspace/daeyong/second_finetuning_data/{name}.json")
	print(len(df))
	df = delete_comma_at_the_end(df)
	df = more_than_1_passage_filter(df)
	df = says_no_information_filter(df)
	df = previous_steps_paraphrase_filter(df)
	df = current_step_paraphrase_filter(df)
	df.to_json(f"/workspace/daeyong/second_finetuning_data/{name}_final.json", orient="records", lines=False, indent=2)

In [ ]:
filtered_rows = []
df = pd.read_json(f"/workspace/daeyong/second_finetuning_data/correct_improved.json")
print(len(df))
for i, row in df.iterrows():
	flag = False
	for step in row['ideal_steps']:
		step_lower = step.lower()
		# 하나라도 금지 문구가 포함되어 있으면 플래그 설정
		if any(phrase in step_lower for phrase in filter_list):
			flag = True
			break

	if not flag:
		filtered_rows.append(row)

filtered_df = pd.DataFrame(filtered_rows)
print(len(filtered_df))
filtered_df.to_json("/workspace/daeyong/second_finetuning_data/correct_improved.json", orient="records", lines=False, indent=2)

In [ ]:
import pandas as pd
dfs = []
for name in ["contradictory", "correct", "inefficiency", "information_miss", "logical_fallacy", "off_topic", "overthinking", "redundancy", "unsupported"]:
	df = pd.read_json(f"/workspace/daeyong/second_finetuning_data/{name}.json")
	dfs.append(df)

final_df = pd.concat(dfs)
print(f"{len(final_df)} before filtering")

print("1. More than 1 passage filter")
filter1 = []
for i, row in final_df.iterrows():
	is_wrong=False
	for step in row["ideal_steps"]:
		passage_refs = []
		matches = re.findall(r"Passage\s+\d+", step)
		passage_refs.extend(matches)

		# Passage 언급이 2개 이상이면 제외
		if len(passage_refs) >= 2:
			is_wrong = True
			# print(f"prev Row {i}: {step}.")
			break

	if is_wrong:
		continue
	filter1.append(row)

filter1_df = pd.DataFrame(filter1)
print(f"{len(filter1_df)} after passage count filtering")

print("2. Says no information filtering")
filter2 = []
for i, row in filter1_df.iterrows():
	flag = False
	for step in row['ideal_steps']:
		step_lower = step.lower()
		# 하나라도 금지 문구가 포함되어 있으면 플래그 설정
		if any(phrase in step_lower for phrase in filter_list):
			flag = True
			break

	if not flag:
		filter2.append(row)

filter2_df = pd.DataFrame(filter2)
print(len(filter2_df))
filter2_df

print("3. attribution step 양식 다양하게")

print(sum(filter2_df['current_step'].apply(lambda x: "According to Passage " in x)))

In [ ]:
import pandas as pd
from collections import Counter

count = 0
dfs = []
for name in ["contradictory", "correct_improved", "inefficiency", "information_miss", "logical_fallacy", "off_topic", "overthinking", "redundancy", "unsupported"]:
	df = pd.read_json(f"/workspace/daeyong/second_finetuning_data/{name}.json")
	index_to_delete = []
	for i, row in df.iterrows():
		new_previous_steps = []
		is_wrong = False
		for j, step in enumerate(row['previous_steps']):
			if step.endswith("(Logical)"):
				new_previous_steps.append(step)
				continue
			if get_paraphrased_text(step):
				new_previous_steps.append(f"Step {j+1}: {get_paraphrased_text(step)}")
			else:
				is_wrong = True
				new_previous_steps.append(step)
		if is_wrong:
			print(f"Warning: Some steps in row {i} could not be paraphrased correctly.")
			count += 1
			# index_to_delete.append(i)
			continue

		df.at[i, 'previous_steps'] = new_previous_steps
		
		error_step_position = len(row['previous_steps'])
	
		# if row['current_step'].endswith("(Logical)"):
		# 	df.at[i, 'current_step'] = row['current_step']
		# else:
		# 	df.at[i, 'current_step'] = f"Step {error_step_position+1}: {get_paraphrased_text(row['current_step'])}"
	dfs.append(df)
    
final_df = pd.concat(dfs)
print(count)
print(Counter(final_df['error_type']))
final_df

In [ ]:
x = final_df[final_df['current_step'].apply(lambda x: "According to Passage " in x)]
print(Counter(x['error_type']))

In [ ]:
index_to_delete = []
for i, row in final_df.iterrows():
	new_previous_steps = []
	is_wrong = False
	for j, step in enumerate(row['previous_steps']):
		if step.endswith("(Logical)"):
			new_previous_steps.append(step)
			continue
		if get_paraphrased_text(step):
			new_previous_steps.append(f"Step {j+1}: {get_paraphrased_text(step)}")
		else:
			is_wrong = True
			new_previous_steps.append(step)
	if is_wrong:
		print(f"Warning: Some steps in row {i} could not be paraphrased correctly.")
		# index_to_delete.append(i)
		continue

	print(new_previous_steps)
	final_df.at[i, 'previous_steps'] = new_previous_steps
	
	error_step_position = len(row['previous_steps'])
 
	if row['current_step'].endswith("(Logical)"):
		final_df.at[i, 'current_step'] = row['current_step']
	else:
		final_df.at[i, 'current_step'] = f"Step {error_step_position+1}: {get_paraphrased_text(row['current_step'])}"

# final_df.drop(index=index_to_delete, inplace=True)
# final_df.reset_index(drop=True, inplace=True)
final_df